# Nettoyage et Enrichissement des Données Kickstarter - Juillet 2025 à Juin 2026

## Informations sur le dataset
Le dataset vient des données scrappées par Web Robots sur la plateforme Kickstarter, entre Juillet 2025 et Juin 2026.
Chaque mois est composé de plusieurs fichiers CSV qu'il conviendra d'abord de fusionner afin d'évaluer convenablement la volumétrie du dataset.

### Nombre de fichiers CSV par mois
* **Juillet 2025** : 81
* **Août 2025** : 83
* **Septembre 2025** : 83
* **Octobre 2025** : 83
* **Novembre 2025** : 83
* **Décembre 2025** : 84
* **Janvier 2026** : 84
* **Février 2026** : 85
* **Mars 2026** : 85
* **Avril 2026** : 86
* **Mai 2026** : 86
* **Juin 2026** : 86

Les fichiers seront fusionnés sur un ETL (Knime).

Inspectons les informations du mois de juin 2026 :

In [253]:
import pandas as pd

raw_202606 = pd.read_csv('/Users/mariamalaborde/Documents/DataScientest/memoire-2026/data/1-merged/202606-kickstarter.csv', sep=',', encoding='utf-8')
raw_202606.info()

<class 'pandas.DataFrame'>
RangeIndex: 272518 entries, 0 to 272517
Data columns (total 42 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   backers_count                       272518 non-null  int64  
 1   blurb                               272396 non-null  str    
 2   category                            272518 non-null  str    
 3   converted_pledged_amount            252269 non-null  float64
 4   country                             272518 non-null  str    
 5   country_displayable_name            272518 non-null  str    
 6   created_at                          272518 non-null  int64  
 7   creator                             272518 non-null  str    
 8   currency                            272518 non-null  str    
 9   currency_symbol                     272518 non-null  str    
 10  currency_trailing_code              272518 non-null  bool   
 11  current_currency                    2

Avec une telle volumétrie de données par mois, il est nécessaire de travailler sur un échantillon.

## Échantillonnage

Pour travailler sur un échantillon sans altérer la distribution entre les projets qui ont réussi et ceux qui ont échoué, nous allons prélever un échantillon de 5% tout en respectant la distribution de la variable `state`.

In [254]:
# Distribution de la variable 'state' sur le mois de Juin 2026
print("Nombre de projets par statut")
print(raw_202606['state'].value_counts())

print("--------------------------------------")
# Répartition en pourcentage
print("Répartition des statuts en pourcentage %")
print(raw_202606['state'].value_counts(normalize=True) * 100)

Nombre de projets par statut
state
successful    158161
failed         76934
submitted      16357
canceled        9671
live            7503
started         3879
suspended         13
Name: count, dtype: int64
--------------------------------------
Répartition des statuts en pourcentage %
state
successful   58.04
failed       28.23
submitted     6.00
canceled      3.55
live          2.75
started       1.42
suspended     0.00
Name: proportion, dtype: float64


In [255]:
# Extraction stratifiée de 5% sur la colonne 'state'
sample_202606 = raw_202606.groupby('state', group_keys=False).sample(frac=0.05, random_state=42)

print(f"Taille de l'échantillon de Juin 2026 (5%) : {len(sample_202606)} lignes")

# Vérification de la distribution
print("Répartition des statuts en pourcentage dans l'échantillon de Juin 2026")
print(sample_202606['state'].value_counts(normalize=True) * 100)

Taille de l'échantillon de Juin 2026 (5%) : 13627 lignes
Répartition des statuts en pourcentage dans l'échantillon de Juin 2026
state
successful   58.03
failed       28.23
submitted     6.00
canceled      3.55
live          2.75
started       1.42
suspended     0.01
Name: proportion, dtype: float64


Cette méthode d'échantillonnage fonctionne et préserve la distribution de la variable 'state'. Reproduisons-la pour chacun des mois du dataset.

In [256]:
# Boucle d'échantillonnage

liste_mois = [ '202507', '202508', '202509', '202510', '202511', '202512', '202601', '202602', '202603', '202604', '202605', '202606']

echantillons_mensuels = []

for mois in liste_mois:
    chemin_fichier = f'/Users/mariamalaborde/Documents/DataScientest/memoire-2026/data/1-merged/{mois}-kickstarter.csv'

    raw_mois = pd.read_csv(chemin_fichier, sep=',', encoding='utf-8')

    sample_mois = raw_mois.groupby('state', group_keys=False).sample(frac=0.05, random_state=42)

    echantillons_mensuels.append(sample_mois)
    print(f"Mois {mois} traité : {len(sample_mois)} lignes extraites.")

Mois 202507 traité : 12886 lignes extraites.
Mois 202508 traité : 13102 lignes extraites.
Mois 202509 traité : 13145 lignes extraites.
Mois 202510 traité : 13119 lignes extraites.
Mois 202511 traité : 13251 lignes extraites.
Mois 202512 traité : 13318 lignes extraites.
Mois 202601 traité : 13314 lignes extraites.
Mois 202602 traité : 13384 lignes extraites.
Mois 202603 traité : 13427 lignes extraites.
Mois 202604 traité : 13493 lignes extraites.
Mois 202605 traité : 13526 lignes extraites.
Mois 202606 traité : 13627 lignes extraites.


In [257]:
# Concaténation de tous les mois
sample_year = pd.concat(echantillons_mensuels, ignore_index=True)

print(f"Taille totale de l'échantillon global (5%) : {len(sample_year)} lignes")
print("Répartition des statuts en pourcentage")
print(sample_year['state'].value_counts(normalize=True) * 100)


Taille totale de l'échantillon global (5%) : 159592 lignes
Répartition des statuts en pourcentage
state
successful   57.57
failed       28.99
submitted     5.68
canceled      3.58
live          2.97
started       1.21
suspended     0.00
Name: proportion, dtype: float64


In [258]:
sample_year.head()

,backers_count,blurb,category,converted_pledged_amount,country,country_displayable_name,created_at,creator,currency,currency_symbol,...,spotlight,staff_pick,state,state_changed_at,static_usd_rate,urls,usd_exchange_rate,usd_pledged,usd_type,video
0,44,Get ready to embark on a journey through the a...,"{""id"":273,""name"":""Playing Cards"",""analytics_na...",15901.00,US,the United States,1696185467,"{""id"":893001491,""name"":""Grim Entertainment"",""s...",USD,$,...,False,False,canceled,1710361353,1.00,"{""web"":{""project"":""https://www.kickstarter.com...",1.00,15901.00,domestic,"{""id"":1267557,""status"":""successful"",""hls"":""htt..."
1,8,"After waking up in an isolated hospital, a gir...","{""id"":302,""name"":""Thrillers"",""analytics_name"":...",576.00,US,the United States,1475774157,"{""id"":749157170,""name"":""Joseph Muhammad"",""is_r...",USD,$,...,False,False,canceled,1480737154,1.00,"{""web"":{""project"":""https://www.kickstarter.com...",1.00,576.00,domestic,"{""id"":727077,""status"":""successful"",""hls"":null,..."
2,6,A short thriller film,"{""id"":302,""name"":""Thrillers"",""analytics_name"":...",172.00,GB,the United Kingdom,1577796173,"{""id"":318316932,""name"":""Luke Foster"",""slug"":""l...",GBP,£,...,False,False,canceled,1579974555,1.30,"{""web"":{""project"":""https://www.kickstarter.com...",1.31,171.62,domestic,NaN
3,53,"Meet eMAKE, the groundbreaking platform that c...","{""id"":334,""name"":""DIY Electronics"",""analytics_...",21016.00,CA,Canada,1722862825,"{""id"":1777660933,""name"":""eMAKE"",""slug"":""emake-...",CAD,$,...,False,False,canceled,1731518035,0.72,"{""web"":{""project"":""https://www.kickstarter.com...",0.72,21242.90,domestic,"{""id"":1319295,""status"":""successful"",""hls"":""htt..."
4,0,A double feature! An experimental webseries an...,"{""id"":291,""name"":""Action"",""analytics_name"":""Ac...",0.00,US,the United States,1458853621,"{""id"":1666965174,""name"":""Spencer James"",""is_re...",USD,$,...,False,False,canceled,1461132613,1.00,"{""web"":{""project"":""https://www.kickstarter.com...",1.00,0.00,domestic,NaN


## Exploration et enrichissement des données

### Valeurs manquantes

In [259]:
sample_year.isna().sum()

backers_count                             0
blurb                                    54
category                                  0
converted_pledged_amount              10995
country                                   0
country_displayable_name                  0
created_at                                0
creator                                   0
currency                                  0
currency_symbol                           0
currency_trailing_code                    0
current_currency                          0
deadline                                  0
disable_communication                     0
fx_rate                                   0
goal                                      0
id                                        0
is_disliked                           27153
is_in_post_campaign_pledging_phase    41695
is_launched                               0
is_liked                              27153
is_starrable                              0
launched_at                     

### Types de données

In [260]:
sample_year.info()

<class 'pandas.DataFrame'>
RangeIndex: 159592 entries, 0 to 159591
Data columns (total 42 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   backers_count                       159592 non-null  int64  
 1   blurb                               159538 non-null  str    
 2   category                            159592 non-null  str    
 3   converted_pledged_amount            148597 non-null  float64
 4   country                             159592 non-null  str    
 5   country_displayable_name            159592 non-null  str    
 6   created_at                          159592 non-null  int64  
 7   creator                             159592 non-null  str    
 8   currency                            159592 non-null  str    
 9   currency_symbol                     159592 non-null  str    
 10  currency_trailing_code              159592 non-null  bool   
 11  current_currency                    1

### Statistiques basiques

In [261]:
sample_year.describe(include='all')

,backers_count,blurb,category,converted_pledged_amount,country,country_displayable_name,created_at,creator,currency,currency_symbol,...,spotlight,staff_pick,state,state_changed_at,static_usd_rate,urls,usd_exchange_rate,usd_pledged,usd_type,video
count,159592.00,159538,159592,148597.00,159592,159592,159592.00,159592,159592,159592,...,159592,159592,159592,159592.00,159592.00,159592,148597.00,148597.00,159532,100246
unique,NaN,111861,189,NaN,25,25,NaN,159539,15,7,...,2,2,7,NaN,NaN,112745,NaN,NaN,2,99058
top,NaN,"A high-quality Figurines, STL file, 3D printab...","{""id"":34,""name"":""Tabletop Games"",""analytics_na...",NaN,US,the United States,NaN,"{""id"":1476467552,""name"":""Lyvvy Art studio"",""sl...",USD,$,...,True,False,successful,NaN,NaN,"{""web"":{""project"":""https://www.kickstarter.com...",NaN,NaN,domestic,"{""id"":1351579,""status"":""successful"",""hls"":""htt..."
freq,NaN,45,5832,NaN,99108,99108,NaN,3,99113,118460,...,91875,135013,91875,NaN,NaN,9,NaN,NaN,159505,3
mean,131.98,NaN,NaN,19258.85,NaN,NaN,1599269824.26,NaN,NaN,NaN,...,NaN,NaN,NaN,1606785458.36,0.91,NaN,0.98,19251.25,NaN,NaN
std,679.39,NaN,NaN,265779.66,NaN,NaN,135505452.80,NaN,NaN,NaN,...,NaN,NaN,NaN,135584974.18,0.36,NaN,0.27,265712.04,NaN,NaN
min,0.00,NaN,NaN,0.00,NaN,NaN,1240366270.00,NaN,NaN,NaN,...,NaN,NaN,NaN,1242468025.00,0.00,NaN,0.01,0.00,NaN,NaN
25%,3.00,NaN,NaN,246.00,NaN,NaN,1476029203.75,NaN,NaN,NaN,...,NaN,NaN,NaN,1482930027.25,1.00,NaN,1.00,246.56,NaN,NaN
50%,26.00,NaN,NaN,2141.00,NaN,NaN,1627261563.00,NaN,NaN,NaN,...,NaN,NaN,NaN,1636625103.00,1.00,NaN,1.00,2142.55,NaN,NaN
75%,88.00,NaN,NaN,8230.00,NaN,NaN,1726656554.50,NaN,NaN,NaN,...,NaN,NaN,NaN,1734699608.00,1.00,NaN,1.00,8230.00,NaN,NaN


### Aperçu de la surperformance

In [262]:
# Enlever notation scientifique

pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [263]:
# On ne veut que les succès.
successful_sample = sample_year[sample_year['state'] == 'successful']

# Top 10 par pourcentage de financement (%)
top10_percent = successful_sample.sort_values(by='percent_funded', ascending=False).head(10)

print("Top 10 : Pourcentage de financement (%)")
col_usd = 'usd_pledged' if 'usd_pledged' in sample_year.columns else 'converted_pledged_amount'
display(top10_percent[['id', 'name', 'percent_funded', col_usd]])

print("-----------------------------------")

# Top 10 par somme récoltée en USD
top10_usd = successful_sample.sort_values(by=col_usd, ascending=False).head(10)

print("Top 10 : Somme récoltée (USD)")
display(top10_usd[['id', 'name', col_usd, 'percent_funded']])

Top 10 : Pourcentage de financement (%)


,id,name,percent_funded,usd_pledged
126317,149159163,AdventureQuest Worlds: Infinity,207752600.00,2077526.00
24634,649845747,Laurent Garnier: Off the Record,15532039.00,195650.15
141533,712425525,LET ME HOLD A DOLLA! | LARUSSELL GOES MAJORLY ...,3942300.00,39423.00
104568,712425525,LET ME HOLD A DOLLA! | LARUSSELL GOES MAJORLY ...,3942300.00,39423.00
50879,712425525,LET ME HOLD A DOLLA! | LARUSSELL GOES MAJORLY ...,3942300.00,39423.00
126996,712425525,LET ME HOLD A DOLLA! | LARUSSELL GOES MAJORLY ...,3942300.00,39423.00
60088,246326560,Tim Rose Visual Album: <3,2868818.00,28688.18
100695,246326560,Tim Rose Visual Album: <3,2868818.00,28688.18
45531,1077219132,THE 'mi8' RISES | The Best Wireless Duo Stereo...,2260300.00,22603.00
11483,956524635,Doom Cat,1176000.00,11760.00


-----------------------------------
Top 10 : Somme récoltée (USD)


,id,name,usd_pledged,percent_funded
138771,1812062326,eufyMake E1: the First Personal 3D-Texture UV ...,46762258.00,9352.45
152972,1812062326,eufyMake E1: the First Personal 3D-Texture UV ...,46762258.00,9352.45
23574,1812062326,eufyMake E1: the First Personal 3D-Texture UV ...,46762258.00,9352.45
77278,411573180,Snapmaker U1 Color 3D Printer: 5X More Speed. ...,20614548.00,20614.55
142499,1850458289,AWOL Vision Aetherion: Pixel-Clarity RGB Laser...,18649456.00,3729.89
156294,1850458289,AWOL Vision Aetherion: Pixel-Clarity RGB Laser...,18649456.00,3729.89
100030,1198848775,EcoFlow DELTA Pro: The Portable Home Battery,12179651.05,12179.65
37196,1431038175,The Smith Blade (21-in-1 Titanium Multi-Tool),10731667.54,5919.77
50189,1404966192,LiberNovo Omni - World's First Dynamic Ergonom...,9101185.04,18231.61
38419,1404966192,LiberNovo Omni - World's First Dynamic Ergonom...,8315670.41,16658.06


### Gestion des doublons

À l'aide de ce classement, on voit cependant qu'il y a des doublons. En effet, Web Robots récolte les données chaque mois. Si un projet dure plusieurs mois et qu'il reste en ligne, il se retrouvera dans plusieurs mois du dataset. En prélevant 5% de chaque fichier avec l'échantillonnage, la probabilité d'extraire plusieurs fois le même projet à des mois différents est très forte.

**Parti Pris :** Garder la dernière version connue de chaque projet.

In [264]:
sample_year_unique = sample_year.drop_duplicates(subset=['id'], keep='last')

print(f"Nombre de lignes avant déduplication : {len(sample_year)}")
print(f"Nombre de projets uniques : {len(sample_year_unique)}")

Nombre de lignes avant déduplication : 159592
Nombre de projets uniques : 112576


Retentons le classement après avoir supprimé les doublons :

In [265]:
successful_sample = sample_year_unique[sample_year_unique['state'] == 'successful']

# Top 10 par pourcentage de financement (%)
top10_percent = successful_sample.sort_values(by='percent_funded', ascending=False).head(10)

print("Top 10 : Pourcentage de financement (%)")
col_usd = 'usd_pledged' if 'usd_pledged' in sample_year.columns else 'converted_pledged_amount'
display(top10_percent[['id', 'name', 'percent_funded', col_usd]])

print("-----------------------------------")

# Top 10 par somme récoltée en USD
top10_usd = successful_sample.sort_values(by=col_usd, ascending=False).head(10)

print("Top 10 : Somme récoltée (USD)")
display(top10_usd[['id', 'name', col_usd, 'percent_funded']])

Top 10 : Pourcentage de financement (%)


,id,name,percent_funded,usd_pledged
126317,149159163,AdventureQuest Worlds: Infinity,207752600.00,2077526.00
24634,649845747,Laurent Garnier: Off the Record,15532039.00,195650.15
141533,712425525,LET ME HOLD A DOLLA! | LARUSSELL GOES MAJORLY ...,3942300.00,39423.00
100695,246326560,Tim Rose Visual Album: <3,2868818.00,28688.18
45531,1077219132,THE 'mi8' RISES | The Best Wireless Duo Stereo...,2260300.00,22603.00
11483,956524635,Doom Cat,1176000.00,11760.00
64718,620302213,LOVELAND Round 6: A Force More Powerful,1000000.00,100.00
6867,1505105087,Chamber Music from Hell,912377.00,9123.77
115805,264728742,Mike Mains & The Branches - When We Were In Love,810051.00,8100.51
99361,1694496142,Big Something Tumbleweed Pre-Order,752900.00,7529.00


-----------------------------------
Top 10 : Somme récoltée (USD)


,id,name,usd_pledged,percent_funded
152972,1812062326,eufyMake E1: the First Personal 3D-Texture UV ...,46762258.00,9352.45
77278,411573180,Snapmaker U1 Color 3D Printer: 5X More Speed. ...,20614548.00,20614.55
156294,1850458289,AWOL Vision Aetherion: Pixel-Clarity RGB Laser...,18649456.00,3729.89
100030,1198848775,EcoFlow DELTA Pro: The Portable Home Battery,12179651.05,12179.65
37196,1431038175,The Smith Blade (21-in-1 Titanium Multi-Tool),10731667.54,5919.77
50189,1404966192,LiberNovo Omni - World's First Dynamic Ergonom...,9101185.04,18231.61
25234,604109210,Snapmaker 2.0: Modular 3-in-1 3D Printers,7850866.96,7850.87
63309,1288458889,Lymow One｜Boundary-Free Robot Mower for Any Te...,7488321.00,24961.07
20119,814466473,Bambu Lab X1: CoreXY Color 3D Printer with Lid...,7003444.17,70030.96
131934,1859235807,UGREEN NASync Series: Your Private Cloud Stora...,6678664.00,33393.32


### Variables temporelles
En vue d'analyser les axes suivants :

* La durée des campagnes a-t-elle un impact sur leur succès ?
* La saisonnalité des lancements, certains mois génèrent-ils de meilleurs taux de financement ?

In [266]:
# Conversion des timestamps unix en datetime

cols_dates = ['deadline', 'launched_at', 'created_at']

for col in cols_dates:
    if col in sample_year_unique.columns:
        # errors='coerce' transforme les éventuelles valeurs corrompues en NaT (vide)
        sample_year_unique[col] = pd.to_datetime(sample_year_unique[col], unit='s', errors='coerce')

print(sample_year_unique[cols_dates].dtypes)

sample_year_unique[['id', 'name', 'created_at', 'launched_at', 'deadline']].head(5)

deadline       datetime64[s]
launched_at    datetime64[s]
created_at     datetime64[s]
dtype: object


,id,name,created_at,launched_at,deadline
2,1606459453,The Village Green - a graduate film (Canceled),2019-12-31 12:42:53,2020-01-21 17:40:57,2020-02-20 17:40:57
4,1963579114,DOUBLE FEATURE! A Webseries w/ a Short Film! (...,2016-03-24 21:07:01,2016-03-24 22:33:53,2016-05-06 21:07:00
5,176076627,Your Life in the Metaverse - Artificial Intell...,2021-12-13 12:53:10,2021-12-17 12:22:53,2022-01-16 12:22:53
6,109080627,"Clever Girls: Stickers, Car Air Fresheners, & ...",2022-06-08 03:30:25,2022-06-14 05:24:28,2022-07-09 05:24:28
7,1737298817,Journey through Spain (Canceled),2019-11-29 09:23:38,2019-11-29 10:38:51,2020-01-28 10:38:51


In [267]:
# Ajout d'une colonne pour montrer la durée en jours
sample_year_unique['duration_days'] = (sample_year_unique['deadline'] - sample_year_unique['launched_at']).dt.days

# Ajout d'une colonne pour montrer la durée de préparation de la campagne
sample_year_unique['prep_days'] = (sample_year_unique['launched_at'] - sample_year_unique['created_at']).dt.days

# Ajout d'une colonne pour voir le mois de lancement
sample_year_unique['launch_month'] = sample_year_unique['launched_at'].dt.month_name()

# Ajout d'une colonne pour voir l'année de lancement
sample_year_unique['launch_year'] = sample_year_unique['launched_at'].dt.year

display(sample_year_unique[['id', 'name', 'duration_days', 'prep_days', 'launch_month', 'launch_year']].head(5))

print(sample_year_unique['launch_year'].value_counts().sort_index())

,id,name,duration_days,prep_days,launch_month,launch_year
2,1606459453,The Village Green - a graduate film (Canceled),30,21,January,2020
4,1963579114,DOUBLE FEATURE! A Webseries w/ a Short Film! (...,42,0,March,2016
5,176076627,Your Life in the Metaverse - Artificial Intell...,30,3,December,2021
6,109080627,"Clever Girls: Stickers, Car Air Fresheners, & ...",25,6,June,2022
7,1737298817,Journey through Spain (Canceled),60,0,November,2019


launch_year
1970     6582
2009       77
2010      413
2011     1169
2012     2198
2013     2501
2014     7507
2015     9729
2016     7394
2017     7348
2018     6193
2019     5729
2020     4970
2021     5059
2022     5633
2023     6579
2024    12759
2025    16654
2026     4082
Name: count, dtype: int64


Certains projets sont vus comme ayant été lancés en 1970. C'est une particularité des timestamp unix : les timestamps démarrent en 1970. Ainsi, si la valeur de la timestamp est à 0 (manquante), alors on verra l'année 1970, ce qui est aberrant car Kickstarter a vu le jour le 28 avril 2009.

In [268]:
# Correction de l'année 1970
# Remplacer les 0 par NaN
import numpy as np
sample_year_unique.loc[sample_year_unique['launched_at'].dt.year == 1970, 'launched_at'] = pd.NaT

# Recalcul des variables temporelles
sample_year_unique['duration_days'] = (sample_year_unique['deadline'] - sample_year_unique['launched_at']).dt.days.astype('Int64')
sample_year_unique['launch_month'] = sample_year_unique['launched_at'].dt.month_name()
sample_year_unique['launch_year'] = sample_year_unique['launched_at'].dt.year.astype('Int64')
sample_year_unique['prep_days'] = (sample_year_unique['launched_at'] - sample_year_unique['created_at']).dt.days.astype('Int64')

# Répartition des lancements par année
print(sample_year_unique['launch_year'].value_counts(dropna=False).sort_index())
print("-----------------------------")

launch_year
2009       77
2010      413
2011     1169
2012     2198
2013     2501
2014     7507
2015     9729
2016     7394
2017     7348
2018     6193
2019     5729
2020     4970
2021     5059
2022     5633
2023     6579
2024    12759
2025    16654
2026     4082
<NA>     6582
Name: count, dtype: Int64
-----------------------------


### Variables financières
En vue d'analyser les axes suivants :

* Comparer la distribution des objectifs fixés entre les projets réussis et échouées. Les projets avec des objectifs ambitieux ont-ils un taux de conversion supérieur ?
* Taux de surfinancement : étudier la distribution de `percent_funded` sur l'ensemble des projets successful pour voir si la surperformance est marginale ou courante.

Nous sommes en France et notre monnaie est l'euro. La première étape est de convertir les variables financières en EUR pour une meilleure lisibilité.

In [269]:
# Calcul d'une variable 'goal_usd' car le goal est en devise d'origine
# Taux implicite USD / devise d'origine
usd_fx = sample_year_unique['usd_pledged'] / sample_year_unique['pledged']

# Si pledged == 0, on utilise fx_rate en fallback
sample_year_unique['goal_usd'] = np.where(
    sample_year_unique['pledged'] > 0,
    sample_year_unique['goal'] * usd_fx,
    sample_year_unique['goal'] * sample_year_unique['fx_rate']
)

# Calcul de 'goal_eur' et 'pledged_eur'
# Taux USD vers EUR au 21/07/2026
usd_to_eur_rate = 0.88

sample_year_unique['goal_eur'] = np.where(
    sample_year_unique['currency'] == 'EUR',
    sample_year_unique['goal'],
    sample_year_unique['goal_usd'] * usd_to_eur_rate
)

sample_year_unique['pledged_eur'] = np.where(
    sample_year_unique['currency'] == 'EUR',
    sample_year_unique['pledged'],
    sample_year_unique['usd_pledged'] * usd_to_eur_rate
)

display(sample_year_unique[['id', 'name', 'goal_eur', 'goal_usd', 'goal', 'currency']].head(5))
print("On voit désormais les montants en euro.")

,id,name,goal_eur,goal_usd,goal,currency
2,1606459453,The Village Green - a graduate film (Canceled),6349.97,7215.87,5550.00,GBP
4,1963579114,DOUBLE FEATURE! A Webseries w/ a Short Film! (...,7040.00,8000.00,8000.00,USD
5,176076627,Your Life in the Metaverse - Artificial Intell...,1748.91,1987.40,1500.00,GBP
6,109080627,"Clever Girls: Stickers, Car Air Fresheners, & ...",132.00,150.00,150.00,USD
7,1737298817,Journey through Spain (Canceled),25500.00,28066.67,25500.00,EUR


On voit désormais les montants en euro.


### Dynamique de communauté
* Identifier le panier moyen `pledged_eur` / `backers_count` par projet
* Définir profil type de backer en fonction du panier moyen

In [270]:
# Calcul du panier moyen par projet (avec sécurité si 0 backer)
sample_year_unique['avg_basket_eur'] = np.where(
    sample_year_unique['backers_count'] > 0,
    sample_year_unique['pledged_eur'] / sample_year_unique['backers_count'],
    0
)

# Calcul des quartiles sur les projets ayant au moins 1 backer / > 0€
valid_pledges = sample_year_unique[sample_year_unique['avg_basket_eur'] > 0]['avg_basket_eur']

# Seuils des quartiles du panier moyen
quartiles = valid_pledges.quantile([0, 0.25, 0.50, 0.75, 1.0])
print(f"Min (0%)    : {quartiles[0.00]:.2f} €")
print(f"Q1  (25%)   : {quartiles[0.25]:.2f} €  -> Micro-mécénat")
print(f"Q2  (50%)   : {quartiles[0.50]:.2f} €  -> Client standard")
print(f"Q3  (75%)   : {quartiles[0.75]:.2f} €  -> Acheteur requis")
print(f"Max (100%)  : {quartiles[1.00]:.2f} €  -> Grand mécène")

Min (0%)    : 0.40 €
Q1  (25%)   : 26.94 €  -> Micro-mécénat
Q2  (50%)   : 50.88 €  -> Client standard
Q3  (75%)   : 91.16 €  -> Acheteur requis
Max (100%)  : 9651.02 €  -> Grand mécène


In [271]:
# Définition des 4 tranches automatiques + 1 tranche dédiée aux 0€
sample_year_unique['backer_profile_qcut'] = pd.qcut(
    valid_pledges,
    q=4,
    labels=['Micro-mécénat (Q1)', 'Client classique (Q2)', 'Acheteur requis (Q3)', 'Grand mécène (Q4)']
)

# Conversion en type 'str' pour ajouter l'étiquette 0€
sample_year_unique['backer_profile_qcut'] = sample_year_unique['backer_profile_qcut'].astype(str)
sample_year_unique.loc[sample_year_unique['avg_basket_eur'] == 0, 'backer_profile_qcut'] = 'Aucun soutien (0€)'

# Vérification
print(sample_year_unique['backer_profile_qcut'].value_counts())

backer_profile_qcut
Micro-mécénat (Q1)       24996
Acheteur requis (Q3)     24996
Grand mécène (Q4)        24996
Client classique (Q2)    24996
Aucun soutien (0€)       12592
Name: count, dtype: int64


### Analyse par Catégorie et Sous-catégorie
La variable `category` est un morceau de JSON dont il faut extraire la catégorie et la sous-catégorie.

* Taux de succès par catégorie
* Montant moyen récolté par catégorie
* `percent_funded` par catégorie

In [272]:
import json

def extract_categories(val):
    if pd.isna(val):
        return pd.Series([None, None])
    
    # Si c'est une chaîne de caractères: dictionnaire
    if isinstance(val, str):
        try:
            val = json.loads(val)
        except:
            return pd.Series([None, None])
            
    if isinstance(val, dict):
        # Catégorie principale : 'parent_name' (fallback sur 'analytics_name' si c'est une catégorie racine sans parent)
        main_cat = val.get('parent_name') or val.get('analytics_name')
        
        # Sous-catégorie : 'analytics_name'
        sub_cat = val.get('analytics_name')
        
        return pd.Series([main_cat, sub_cat])
        
    return pd.Series([None, None])

# Application du découpage sur le DataFrame
sample_year_unique[['main_category', 'sub_category']] = sample_year_unique['category'].apply(extract_categories)

In [273]:
display(sample_year_unique[['id', 'name', 'main_category', 'sub_category']].head(5))

,id,name,main_category,sub_category
2,1606459453,The Village Green - a graduate film (Canceled),Film & Video,Thrillers
4,1963579114,DOUBLE FEATURE! A Webseries w/ a Short Film! (...,Film & Video,Action
5,176076627,Your Life in the Metaverse - Artificial Intell...,Publishing,Radio & Podcasts
6,109080627,"Clever Girls: Stickers, Car Air Fresheners, & ...",Art,Art
7,1737298817,Journey through Spain (Canceled),Photography,Places


In [274]:
# Taux de succès moyen par catégorie pour comparer chaque projet à sa catégorie en pourcentage
sample_year_unique['cat_mean_success_rate'] = sample_year_unique.groupby('main_category')['state'].transform(
    lambda x: (x == 'successful').mean() * 100
)

# Calcul de la médiane des montants récoltés en EUR par catégorie principale
sample_year_unique['cat_median_pledged_eur'] = sample_year_unique.groupby('main_category')['pledged_eur'].transform('median')

# Surperformance du projet vs sa catégorie
# Est-ce que le projet a récolté plus que la médiane de sa propre catégorie ?
sample_year_unique['outperformed_cat_median_eur'] = sample_year_unique['pledged_eur'] > sample_year_unique['cat_median_pledged_eur']

display(sample_year_unique[['id', 'name', 'main_category', 'sub_category', 'cat_median_pledged_eur', 'outperformed_cat_median_eur', 'cat_mean_success_rate']].head(5))

,id,name,main_category,sub_category,cat_median_pledged_eur,outperformed_cat_median_eur,cat_mean_success_rate
2,1606459453,The Village Green - a graduate film (Canceled),Film & Video,Thrillers,2047.26,False,60.79
4,1963579114,DOUBLE FEATURE! A Webseries w/ a Short Film! (...,Film & Video,Action,2047.26,False,60.79
5,176076627,Your Life in the Metaverse - Artificial Intell...,Publishing,Radio & Podcasts,1551.13,False,56.08
6,109080627,"Clever Girls: Stickers, Car Air Fresheners, & ...",Art,Art,1390.40,False,68.61
7,1737298817,Journey through Spain (Canceled),Photography,Places,964.56,False,55.69


### Dimension géographique
* Analyse de la provenance des projets
* Taux de succès selon la géographie

Pour la vue macro, on peut déjà chercher à enrichir en découpant par région du monde.

In [275]:
# A. Dictionnaire de regroupement par grandes zones économiques
def map_country_to_region(country_code):
    if pd.isna(country_code):
        return 'Non renseigné'
    
    code = str(country_code).upper().strip()
    
    if code in ['CA', 'MX', 'US']:
        return 'Amérique du Nord'
    elif code in ['GB', 'FR', 'DE', 'IT', 'ES', 'NL', 'BE', 'CH', 'AT', 'SE', 'DK', 'NO', 'FI', 'IE', 'PL', 'LU']:
        return 'Europe'
    elif code in ['AU', 'NZ']:
        return 'Océanie'
    elif code in ['HK', 'SG', 'JP']:
        return 'Asie'
    else:
        return 'Reste du monde'

# Application de la région
sample_year_unique['region'] = sample_year_unique['country'].apply(map_country_to_region)

# Taux de succès moyen calculé par Région
sample_year_unique['region_success_rate'] = sample_year_unique.groupby('region')['state'].transform(
    lambda x: (x == 'successful').mean() * 100
)

# 🧪 Vérification des résultats
print("=== RÉPARTITION DES PROJETS PAR RÉGION ===")
print(sample_year_unique['region'].value_counts())

=== RÉPARTITION DES PROJETS PAR RÉGION ===
region
Amérique du Nord    77886
Europe              27221
Asie                 3708
Océanie              3350
Reste du monde        411
Name: count, dtype: int64


### Théorie du Signal
La théorie du signal (Spence, 1973) stipule que dans une situation d'asymétrie d'information (comme Kickstarter, où le backer ne sait pas si le porteur de projet est sérieux), le créateur doit envoyer des signaux de qualité pour rassurer les contributeurs.

* Présence ou non d'une vidéo explicative
* Longueur du blurb et du nom

In [276]:
# Signal Vidéo : Création du flag booléen `has_video`
# Selon le format du dump (dict, string JSON, URL ou None), on sécurise le test :
def check_has_video(val):
    if pd.isna(val) or val is None or val == '' or val == False:
        return False
    # Si c'est un dictionnaire/JSON ou une chaîne non vide
    return True

sample_year_unique['has_video'] = sample_year_unique['video'].apply(check_has_video)


# Signaux Textuels : Longueur en caractères et en mots (Name & Blurb)
# On gère les éventuelles valeurs nulles avec fillna('')
sample_year_unique['name_len_char'] = sample_year_unique['name'].fillna('').astype(str).str.len()
sample_year_unique['name_len_words'] = sample_year_unique['name'].fillna('').astype(str).str.split().str.len()

sample_year_unique['blurb_len_char'] = sample_year_unique['blurb'].fillna('').astype(str).str.len()
sample_year_unique['blurb_len_words'] = sample_year_unique['blurb'].fillna('').astype(str).str.split().str.len()


# Indicateur de clarté/densité (Moyenne de caractères par mot dans le blurb)
sample_year_unique['blurb_density'] = np.where(
    sample_year_unique['blurb_len_words'] > 0,
    sample_year_unique['blurb_len_char'] / sample_year_unique['blurb_len_words'],
    0
)

In [277]:
cols_signal = [
    'name', 'has_video', 
    'name_len_words', 'blurb_len_char', 'blurb_len_words', 
    'blurb_density'
]
display(sample_year_unique[cols_signal].head(10))

,name,has_video,name_len_words,blurb_len_char,blurb_len_words,blurb_density
2,The Village Green - a graduate film (Canceled),False,8,21,4,5.25
4,DOUBLE FEATURE! A Webseries w/ a Short Film! (...,False,9,102,18,5.67
5,Your Life in the Metaverse - Artificial Intell...,True,10,131,18,7.28
6,"Clever Girls: Stickers, Car Air Fresheners, & ...",False,10,59,9,6.56
7,Journey through Spain (Canceled),False,4,37,7,5.29
8,Gli Acrobati (Canceled),True,3,115,15,7.67
9,E1 : All-in-one Lightning Active Noise-cancell...,True,8,127,15,8.47
10,Factallacy Poetry (Canceled),False,3,43,6,7.17
11,GYAAN 2.0... AN INDIAN DANCE SHOW (Canceled),True,7,114,19,6.00
13,Plymouth Film Festival 2016 (Canceled),True,5,134,20,6.70


## Tri des variables

In [278]:
# Liste des colonnes conservées pour l'analyse
cols_to_keep = [
    # Identifiants & Succès
    'id', 'name', 'blurb', 'state', 
    
    # Temporalité
    'created_at', 'launched_at', 'deadline', 
    'duration_days', 'prep_days', 'launch_month', 'launch_year',
    
    # Métriques Financières
    'goal_eur', 'pledged_eur', 'percent_funded', 
    
    # Communauté
    'backers_count', 'avg_basket_eur', 'backer_profile_qcut', 'prelaunch_activated',
    
    # Catégories
    'main_category', 'sub_category', 'cat_median_pledged_eur',
    
    # Géographie
    'country', 'region',
    
    # Théorie du Signal & Marketing
    'has_video', 'staff_pick', 'spotlight',
    'name_len_words', 'name_len_char', 'blurb_len_char', 'blurb_len_words', 'blurb_density'
]


sample_year_unique = sample_year_unique[cols_to_keep].copy()

# Vérification du résultat
print(f"Nouveau nombre de colonnes : {sample_year_unique.shape[1]}")
sample_year_unique.info()

Nouveau nombre de colonnes : 31
<class 'pandas.DataFrame'>
Index: 112576 entries, 2 to 159591
Data columns (total 31 columns):
 #   Column                  Non-Null Count   Dtype        
---  ------                  --------------   -----        
 0   id                      112576 non-null  int64        
 1   name                    112575 non-null  str          
 2   blurb                   112538 non-null  str          
 3   state                   112576 non-null  str          
 4   created_at              112576 non-null  datetime64[s]
 5   launched_at             105994 non-null  datetime64[s]
 6   deadline                112576 non-null  datetime64[s]
 7   duration_days           105994 non-null  Int64        
 8   prep_days               105994 non-null  Int64        
 9   launch_month            105994 non-null  str          
 10  launch_year             105994 non-null  Int64        
 11  goal_eur                112576 non-null  float64      
 12  pledged_eur             1069

## Deuxième passe sur les valeurs manquantes

In [279]:
# Diagnostic des valeurs manquantes
missing_summary = pd.DataFrame({
    'Manquants (N)': sample_year_unique.isnull().sum(),
    'Pourcentage (%)': (sample_year_unique.isnull().sum() / len(sample_year_unique) * 100).round(2)
})

# Filtrer pour n'afficher que les colonnes qui ont au moins 1 valeur manquante
missing_summary = missing_summary[missing_summary['Manquants (N)'] > 0].sort_values(by='Manquants (N)', ascending=False)

print("Valeurs manquantes")
display(missing_summary)

Valeurs manquantes


,Manquants (N),Pourcentage (%)
launched_at,6582,5.85
duration_days,6582,5.85
prep_days,6582,5.85
launch_month,6582,5.85
launch_year,6582,5.85
pledged_eur,5597,4.97
blurb,38,0.03
name,1,0.00


Les valeurs manquantes pour les variables temporelles sont des projets qui n'ont jamais été lancés. Pour analyser les facteurs de succès d'un projet en vue de valider un Product Market Fit, les projets doivent à minima avoir été lancés : il faut donc supprimer ces lignes.

In [280]:
# Suppression des projets non lancés
sample_year_unique.dropna(subset=['launched_at'], inplace=True)

# Diagnostic des valeurs manquantes
missing_summary = pd.DataFrame({
    'Manquants (N)': sample_year_unique.isnull().sum(),
    'Pourcentage (%)': (sample_year_unique.isnull().sum() / len(sample_year_unique) * 100).round(2)
})

# Filtrer pour n'afficher que les colonnes qui ont au moins 1 valeur manquante
missing_summary = missing_summary[missing_summary['Manquants (N)'] > 0].sort_values(by='Manquants (N)', ascending=False)

print("Valeurs manquantes")
display(missing_summary)

Valeurs manquantes


,Manquants (N),Pourcentage (%)
blurb,5,0.00
name,1,0.00


Les valeurs manquantes qui restent sont des projets pour lesquels il n'y a eu ni de blurb, ni de titre. On va tout simplement remplir par "Sans titre" ou "Sans blurb". Le fait de ne pas renseigner ni titre ni blurb est sans doute un facteur qui influence le succès d'une campagne.

In [281]:
# Masques des valeurs initialement manquantes
missing_blurb_mask = sample_year_unique['blurb'].isna()
missing_name_mask = sample_year_unique['name'].isna()

# Imputation du texte pour l'affichage
sample_year_unique['blurb'] = sample_year_unique['blurb'].fillna('Sans blurb')
sample_year_unique['name'] = sample_year_unique['name'].fillna('Sans titre')

# Correction forcée à 0 pour les lignes initialement manquantes
sample_year_unique.loc[missing_blurb_mask, ['blurb_len_char', 'blurb_len_words']] = 0
sample_year_unique.loc[missing_name_mask, ['name_len_char', 'name_len_words']] = 0

# Sécurisation de la longueur moyenne par mot
sample_year_unique['blurb_avg_word_len'] = np.where(
    sample_year_unique['blurb_len_words'] > 0,
    sample_year_unique['blurb_len_char'] / sample_year_unique['blurb_len_words'],
    0
)


In [282]:
# Vérification pour les blurbs initialement manquants
check_blurb = sample_year_unique[sample_year_unique['blurb'] == 'Sans blurb']
display(check_blurb[['name', 'blurb', 'blurb_len_char', 'blurb_len_words', 'blurb_avg_word_len']])

# Vérification pour le name initialement manquant
check_name = sample_year_unique[sample_year_unique['name'] == 'Sans titre']
display(check_name[['name', 'name_len_char', 'name_len_words']])

,name,blurb,blurb_len_char,blurb_len_words,blurb_avg_word_len
17624,Sans titre,Sans blurb,0,0,0.00
52402,N/A (Canceled),Sans blurb,0,0,0.00
79252,Star Wars Bluetooth Speakers (Canceled),Sans blurb,0,0,0.00
140750,Omaxs VentoGo X1:Your 4-in-1 Car Detailing & E...,Sans blurb,0,0,0.00
153506,Cammy The Colorful Chameleon,Sans blurb,0,0,0.00


,name,name_len_char,name_len_words
17624,Sans titre,0,0


In [283]:
# Diagnostic des valeurs manquantes
missing_summary = pd.DataFrame({
    'Manquants (N)': sample_year_unique.isnull().sum(),
    'Pourcentage (%)': (sample_year_unique.isnull().sum() / len(sample_year_unique) * 100).round(2)
})

# Filtrer pour n'afficher que les colonnes qui ont au moins 1 valeur manquante
missing_summary = missing_summary[missing_summary['Manquants (N)'] > 0].sort_values(by='Manquants (N)', ascending=False)

print("Valeurs manquantes")
display(missing_summary)

Valeurs manquantes


,Manquants (N),Pourcentage (%)


Il n'y a plus de valeur manquante.

## Arrondir les floats

In [284]:
float_cols = sample_year_unique.select_dtypes(include=['float64', 'float32']).columns
sample_year_unique[float_cols] = sample_year_unique[float_cols].round(2)

## Export du dataset enrichi et nettoyé

In [285]:
sample_year_unique.to_csv('../data/kickstarter_sample_year_cleaned.csv', index=False)